# Machine Translation (German → English) - Seq2Seq with Attention

In [ ]:
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
from typing import Iterable, List
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

In [ ]:
SRC_LANGUAGE = 'de'
TGT_LANGUAGE = 'en'

In [ ]:
# Setup tokenizers
de_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')
en_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')

print("Tokenizers initialized")

In [ ]:
def yield_tokens(data_iter: Iterable, language: str) -> List[str]:
    for data_sample in data_iter:
        if language == 'de':
            yield de_tokenizer(data_sample[0])
        elif language == 'en':
            yield en_tokenizer(data_sample[1])

UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

In [ ]:
print("Building vocabularies...")
train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))

vocab_de = build_vocab_from_iterator(
    yield_tokens(train_iter, 'de'),
    min_freq=1,
    specials=special_symbols,
    special_first=True
)
vocab_de.set_default_index(UNK_IDX)

train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
vocab_en = build_vocab_from_iterator(
    yield_tokens(train_iter, 'en'),
    min_freq=1,
    specials=special_symbols,
    special_first=True
)
vocab_en.set_default_index(UNK_IDX)

print(f"German vocab: {len(vocab_de)}, English vocab: {len(vocab_en)}")

In [ ]:
def collate_fn(batch):
    src_batch, tgt_batch, src_len = [], [], []
    
    for src_sample, tgt_sample in batch:
        src_sample = src_sample.rstrip("\n")
        tgt_sample = tgt_sample.rstrip("\n")
        
        src_tokens = de_tokenizer(src_sample)
        tgt_tokens = en_tokenizer(tgt_sample)
        
        src_ids = vocab_de(src_tokens)
        tgt_ids = vocab_en(tgt_tokens)
        
        src_ids.append(EOS_IDX)
        tgt_ids.append(EOS_IDX)
        tgt_ids.insert(0, BOS_IDX)
        
        src_len.append(len(src_ids))
        
        src_tensor = torch.tensor(src_ids)
        tgt_tensor = torch.tensor(tgt_ids)
        
        src_batch.append(src_tensor)
        tgt_batch.append(tgt_tensor)
    
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)
    
    return src_batch, tgt_batch, src_len

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_size, embed_size, hidden_size, dropout_p=0.1):
        super().__init__()
        self.e = nn.Embedding(input_size, embed_size)
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(embed_size, hidden_size, batch_first=True)
    
    def forward(self, x, lengths):
        x = self.e(x)
        x = self.dropout(x)
        x = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        outputs, hidden = self.gru(x)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        return outputs, hidden

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_size, embed_size, hidden_size):
        super().__init__()
        self.e = nn.Embedding(output_size, embed_size)
        self.dropout = nn.Dropout()
        self.gru = nn.GRU(embed_size + hidden_size, hidden_size, batch_first=True)
        self.lin = nn.Linear(hidden_size, output_size)
        self.lsoftmax = nn.LogSoftmax(dim=-1)
    
    def forward(self, x, context, prev_hidden):
        x = self.e(x)
        x = self.dropout(x)
        x = torch.cat((x, context), dim=2)
        output, hidden = self.gru(x, prev_hidden)
        y = self.lin(output)
        y = self.lsoftmax(y)
        return y, hidden

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, encoder_hidden_size, decoder_hidden_size, new_hidden_size):
        super().__init__()
        self.eh2nh = nn.Linear(in_features=encoder_hidden_size, out_features=new_hidden_size)
        self.dh2nh = nn.Linear(in_features=decoder_hidden_size, out_features=new_hidden_size)
        self.score = nn.Linear(in_features=new_hidden_size, out_features=1)
    
    def forward(self, query, keys):
        query = self.dh2nh(query)
        keys = self.eh2nh(keys)
        att_score = self.score(torch.tanh(query.permute(1, 0, 2) + keys))
        att_score = att_score.squeeze(2).unsqueeze(1)
        att_weights = F.softmax(att_score, dim=-1)
        context = torch.bmm(att_weights, keys)
        return context, att_weights

In [ ]:
def train_one_epoch(encoder, decoder, attention, train_dataloader, opte, optd, optba, loss_fn, device, max_length=50):
    encoder.train()
    decoder.train()
    attention.train()
    track_loss = 0
    
    for i, (s_ids, t_ids, s_l) in enumerate(train_dataloader):
        s_ids = s_ids.to(device)
        t_ids = t_ids.to(device)
        
        encoder_outputs, encoder_hidden = encoder(s_ids, s_l)
        decoder_hidden = encoder_hidden
        
        yhats = []
        for j in range(min(t_ids.shape[1] - 1, max_length)):
            context, att_weights = attention(decoder_hidden, encoder_outputs)
            probs, decoder_hidden = decoder(t_ids[:, j].unsqueeze(1), context, decoder_hidden)
            yhats.append(probs)
        
        yhats_cat = torch.cat(yhats, dim=1)
        yhats_reshaped = yhats_cat.view(-1, yhats_cat.shape[-1])
        
        gt = t_ids[:, 1:len(yhats)+1]
        gt = gt.reshape(-1)
        
        loss = loss_fn(yhats_reshaped, gt)
        track_loss += loss.item()
        
        opte.zero_grad()
        optd.zero_grad()
        optba.zero_grad()
        
        loss.backward()
        
        opte.step()
        optd.step()
        optba.step()
    
    return track_loss / (i + 1)

In [ ]:
def eval_one_epoch(encoder, decoder, attention, val_dataloader, loss_fn, device, vocab_de, vocab_en, e, n_epochs, max_length=50):
    encoder.eval()
    decoder.eval()
    attention.eval()
    track_loss = 0
    
    with torch.no_grad():
        for i, (s_ids, t_ids, s_l) in enumerate(val_dataloader):
            s_ids = s_ids.to(device)
            t_ids = t_ids.to(device)
            
            encoder_outputs, encoder_hidden = encoder(s_ids, s_l)
            decoder_hidden = encoder_hidden
            input_id = t_ids[:, 0]
            yhats = []
            
            if e + 1 == n_epochs:
                pred_sentence = ""
            
            for j in range(1, min(t_ids.shape[1], max_length)):
                context, att_weights = attention(decoder_hidden, encoder_outputs)
                probs, decoder_hidden = decoder(input_id.unsqueeze(1), context, decoder_hidden)
                yhats.append(probs)
                _, input_id = torch.topk(probs, 1, dim=-1)
                input_id = input_id.squeeze(1, 2)
                
                if e + 1 == n_epochs:
                    word = vocab_en.lookup_token(input_id.item())
                    pred_sentence += word + " "
                
                if input_id.item() == EOS_IDX:
                    break
            
            if e + 1 == n_epochs and i < 5:
                src_sentence_tokens = vocab_de.lookup_tokens(s_ids.tolist()[0])
                src_sentence = " ".join(src_sentence_tokens)
                gt_sentence_tokens = vocab_en.lookup_tokens(t_ids[:, 1:].tolist()[0])
                gt_sentence = " ".join(gt_sentence_tokens)
                print("\n" + "-" * 50)
                print(f"Source: {src_sentence}")
                print(f"Target: {gt_sentence}")
                print(f"Predicted: {pred_sentence}")
            
            yhats_cat = torch.cat(yhats, dim=1)
            yhats_reshaped = yhats_cat.view(-1, yhats_cat.shape[-1])
            gt = t_ids[:, 1:j+1]
            gt = gt.view(-1)
            
            loss = loss_fn(yhats_reshaped, gt)
            track_loss += loss.item()
    
    if e + 1 == n_epochs:
        print("-" * 50)
    
    return track_loss / (i + 1)

In [ ]:
# Hyperparameters
embed_size = 300
hidden_size = 512
batch_size = 32
n_epochs = 10
lr = 0.001

# Initialize models
encoder = Encoder(len(vocab_de), embed_size, hidden_size).to(device)
decoder = Decoder(len(vocab_en), embed_size, hidden_size).to(device)
attention = BahdanauAttention(hidden_size, hidden_size, hidden_size).to(device)

# Loss and optimizers
loss_fn = nn.NLLLoss(ignore_index=PAD_IDX).to(device)
opte = optim.Adam(params=encoder.parameters(), lr=lr)
optd = optim.Adam(params=decoder.parameters(), lr=lr)
optba = optim.Adam(params=attention.parameters(), lr=lr)

print(f"Models initialized on {device}")
print(f"German vocab: {len(vocab_de)}, English vocab: {len(vocab_en)}")

In [ ]:
# Training loop
print("\nStarting training...\n")

for e in range(n_epochs):
    train_iter = datasets.Multi30k(split='train', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    train_dataloader = DataLoader(train_iter, batch_size=batch_size, collate_fn=collate_fn)
    
    val_iter = datasets.Multi30k(split='valid', language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    val_dataloader = DataLoader(val_iter, batch_size=1, collate_fn=collate_fn)
    
    print(f"Epoch {e+1}/{n_epochs}", end=", ")
    
    train_loss = train_one_epoch(encoder, decoder, attention, train_dataloader, opte, optd, optba, loss_fn, device)
    print(f"Train Loss: {train_loss:.4f}", end=", ")
    
    eval_loss = eval_one_epoch(encoder, decoder, attention, val_dataloader, loss_fn, device, vocab_de, vocab_en, e, n_epochs)
    print(f"Val Loss: {eval_loss:.4f}")

## Notes

**Attention Mechanism:**
- Bahdanau-style attention
- Allows decoder to focus on relevant parts of source sentence
- Improves translation quality, especially for longer sentences

**Further Improvements:**
- Bidirectional encoder
- Multi-head attention (Transformer-style)
- Beam search decoding
- Better regularization